1. Tokenization
2. Embeddings (with scaling — you already have this)
3. Positional Encoding (sinusoidal — yours is great, small fix needed)
4. Multi-Head Self-Attention (from scratch)
5. Feed-Forward Network
6. Full Encoder Layer (with residuals + LayerNorm)
7. Stack of N Encoder Layers
8. Classification Head (using [CLS] token)
9. Training loop
10. Validation



1. Tokenization

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding
from datasets import load_dataset
from torch.utils.data import DataLoader

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
ds = load_dataset("stanfordnlp/sst2")


def preprocess(example):
    return tokenizer(example["sentence"], truncation=True, max_length=128)

tokenized_ds = ds.map(preprocess, batched=True)
tokenized_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(tokenized_ds["train"], batch_size=32, shuffle=True, collate_fn=data_collator)
val_loader = DataLoader(tokenized_ds["validation"], batch_size=32, collate_fn=data_collator)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

1. Embeddings

Transformers (and most NLP models) can't work directly with integer token IDs like 5, 42, 999.We need to convert each discrete token (word or subword) into a dense vector of floating-point numbers that the neural network can process and update during training.


In [10]:
import math
import torch
import torch.nn as nn

class Embeddings(nn.Module):
    def __init__(self, vocab_size, d_model) -> None:
        super().__init__()

        self.lookup_table = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.lookup_table(x) * math.sqrt(self.d_model)



8. Training loop



In [22]:
def training_loop():
    batch = next(iter(train_loader))
    input_ids = batch["input_ids"][0]
    mask = batch['attention_mask'][0]
    actual_len = mask.sum().item()
    print("Token ids", actual_len)

    text = tokenizer.decode(input_ids[:actual_len])
    print(text)
    


training_loop()

Token ids 6
[CLS] run out of gas [SEP]
